# Chapter 3: Advanced Topics and Additional Learning Resources

Welcome to Chapter 3, the final chapter of the course 5 minutes to Federated Learning with NVIDIA FLARE!

In [chapter 1](Chapter_1_Intro_APIs_and_Simulator.ipynb) and [chapter 2](Chapter_2_Provision_PoC_Mode_and_Monitoring.ipynb), we learned how to develop federated applications using NVIDIA FLARE's APIs, run them in a simulated environment, as well as how to provision a federated system to run federated applications in NVIDIA FLARE's proo-of-concept mode and monitor runtime metrics. While these are foundational skills to implement federated projects, there are more aspects in NVIDIA FLARE to bring federated projects to real-world production at scale. 

In this final chapter of the course, we aim to introduce some advanced topics in NVIDIA FLARE for real-world deployment. These advanced topics include:  
- Security: privacy preserving technologies, site policy management, Confidential Compute etc.
- FLARE Dashboard
- Federated Large Language Models
- Flower Integration

We will also share references for additional learning resources for you to become a full-fledged federated learning developer.

This chapter will only provide general overview, without hands-on coding and exercises.

After this chapter, you will:
- Have a high-level understand of FLARE's advanced features for real-world deployment of federated applications. 
- Know how and where to search and find guidelines to use or implement FLARE's advanced features.
- Have access to all the learning resources if you aim to master NVIDIA FLARE or maybe become a FLARE developer.


# Security

Security is an essential element in real-world deployment of federated application. NVIDIA FLARE provides many advanced [security features](https://nvidia.github.io/NVFlare/security/) to ensure that a federated application can be bullet-proof in regard to attacks and information leakage.

Below, we briefly introduce privacy preserving technologies, site policy management and Confidential Computing in FLARE.

### Privacy Preserving Technologies

[Privacy-preserving technologies](https://en.wikipedia.org/wiki/Privacy-enhancing_technologies) are technologies that embody fundamental data protection principles by minimizing personal data use, maximizing data security, and empowering individuals. In federated learning, various privacy preserving methods can be employed to protect sensitive data while enabling collaborative model training. Commonly used methods include [differential privacy](https://en.wikipedia.org/wiki/Differential_privacy), [homomorphic encryption](https://en.wikipedia.org/wiki/Homomorphic_encryption) etc.

In NVIDIA FLARE, most privacy preserving methods are implemented leveraging the powerful [filtering mechanism](https://nvflare.readthedocs.io/en/main/programming_guide/filters.html#filters) (remember the FLARE architecture diagram in [chapter 1](Chapter_1_Intro_APIs_and_Simulator.ipynb#NVIDIA-FLARE-Architecture) that shows task filtering mechanism?). As a matter of fact, the filtering mechanism in FLARE offers general [data privacy protection](https://nvflare.readthedocs.io/en/main/user_guide/security/data_privacy_protection.html), with flexibility to add any filtering to any moment of inbound and outbound data exchange between both the server Controllers and client executors. While FLARE provides reference implementations of privacy preserving filters, you can also [write your own custom filters leveraging FLARE's Data Exchange Object class](https://nvflare.readthedocs.io/en/main/programming_guide/filters.html#creating-a-dxo-filter).

<img src="../images/filtering.png" alt="Filtering" width=50% />

Below, we briefly describe differential privacy and homomorphic encryption in FLARE. Refer to this [security page](https://nvidia.github.io/NVFlare/security/) for more details.

- **Differential Privacy**

[Differential Privacy](https://en.wikipedia.org/wiki/Differential_privacy) is a mathematically rigorous framework for protecting individual data privacy while allowing statistical analysis of sensitive data. Common differential privacy algorithms are essentially adding controlled noise to data or model updates. They ensure that the inclusion or exclusion of an individual's data does not significantly affect the results of analyses. Differential privacy offers strong privacy guarantees while still allowing useful insights to be drawn from data, making it a powerful tool for privacy-preserving data analysis. 

FLARE provides a reference implementation of [differential privacy filter using the Sparse Vector Technique](https://nvflare.readthedocs.io/en/main/apidocs/nvflare.app_common.filters.svt_privacy.html). You can also custom your own differential privacy filter.

You can refer to [this example](https://github.com/NVIDIA/NVFlare/tree/main/examples/advanced/brats18) to learn how to use differential privacy in FLARE.

- **Homomorphic Encryption**

[Homomorphic encryption](https://en.wikipedia.org/wiki/Homomorphic_encryption) is a cryptographic method that allows computations to be performed on encrypted data without decrypting it first. It enables mathematical operations on cipher text, producing encrypted results that, when decrypted, match the results of performing the same operations on the original plain text. Homomorphicencryption allows for secure aggregation of model updates and protects data privacy during transmission and aggregation.

In NVIDIA FLARE, similar to differential privacy homomorphic encryption is implemented as a filter, to [encrypt data](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_opt/he/model_encryptor.py#L33) before sharing and to [decrypt data](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_opt/he/model_decryptor.py#L35). Internally, NVIDIA FLARE uses the [TenSEAL](https://github.com/OpenMined/TenSEAL) library, which is a Python wrapper around [Microsoft SEAL](https://github.com/Microsoft/SEAL), for implementing homomorphic encryption.

You can find many examples illustrating how to use homomorphic encryption with NVIDIA FLARE, for instance, [here](https://github.com/NVIDIA/NVFlare/blob/main/examples/hello-world/step-by-step/cifar10/sag_he/sag_he.ipynb) and [here](https://github.com/NVIDIA/NVFlare/blob/main/examples/advanced/cifar10/cifar10-real-world/README.md).

Privacy preserving filters can be enforced at job-level, to the server or clients by modifying the `task_result_filters` and `task_data_filters` sections of corresponding configuration files ([config_fed_server.json](https://nvflare.readthedocs.io/en/main/real_world_fl/application.html#fl-server-configuration) and [config_fed_client.json](https://nvflare.readthedocs.io/en/main/real_world_fl/application.html#fl-client-configuration)) in an exported federated job. Filters can also be applied at site-level, with [site-specific configuration](https://nvflare.readthedocs.io/en/main/user_guide/security/site_policy_management.html#privacy-management). We will see more details on this in the next section.

### Site Policy Management

As we have seen together in [chapter 2](Chapter_2_Provision_PoC_Mode_and_Monitoring.ipynb#2.-Generate-startup-kits), inside each site's startup kit, there is a `local` folder with default site-specific configuration. This design allow each site to define its own policies in the following areas:
- Resource Management: the configuration of system resources that are solely the decisions of local IT.
- Authorization Policy: local authorization policy that determines what a user can or cannot do on the local site.
- Privacy Policy: local policy that specifies what types of studies are allowed and how to add privacy protection to the learning results produced by the clients on the local site.
- Logging Configuration: each site can define its own logging configuration for system generated log messages.

For more details on site policy management, refer to the [dedicated documentation page](https://nvflare.readthedocs.io/en/main/user_guide/security/site_policy_management.html).

### Confidential Computing

[Confidential Computing](https://en.wikipedia.org/wiki/Confidential_computing) is an advanced protects data while it's in use by performing computations in a hardware-based, attested Trusted Execution Environment (TEE). At its core, the technology creates an isolated, encrypted computing environment within a processor that prevents unauthorized access or modification of data and applications while they are being processed. Modern confidential computing implementations provide hardware-level isolation through specialized processor features, leveraging cryptographic techniques to establish a secure perimeter around computational workloads, creating a "black box" where sensitive operations occur. 

The attestation process is a critical component of confidential computing. It provides cryptographic evidence that the computing environment is genuinely secure and has not been compromised. This allows organizations to verify the integrity of the computational environment before transmitting sensitive data. By protecting data during computation, confidential computing addresses a significant security gap that traditional encryption methods couldn't resolve, offering unprecedented levels of data protection across distributed computing environments.

Confidential computing in NVFlare is designed to explicitly establish the trust between participants. Each participant must first capture the evidences related to the hardware (such as GPU), the software (GPU driver and VBIOS) and other components in its own platform. The evidences will be validated and signed to ensure its validity and authenticity. The owner of signed evidences, called confidential computing token (CC token), can demonstrate the information about its computing environment to other participants by providing the CC token. Upon receiving CC token, the participant (the relying party) can verify the claims inside the CC token against it own security policy on whether the CC token owner is using required hardware/software/components for security. If the relying party finds that the CC token does not meet its security policy, the relying party can inform the system that it chooses not to join the job deployment and will not exchange models with others. Only participants who trust and is trusted by one another will work together to run federated jobs.

For more details on Confidential Computing in FLARE, refer to this [dedicated documentation page](https://nvflare.readthedocs.io/en/main/user_guide/confidential_computing.html). You can also learn more about GPU-based Confidential Computing in FLARE by watching [this video](https://developer.download.nvidia.com/assets/Clara/flare/NVFLARE_DAY_2024_Part_13_Confidential_Computing_Closing.mp4).

There are many other security features in NVIDIA FLARE. For more details, please refer to this dedicated [security documentation](https://nvflare.readthedocs.io/en/main/user_guide/nvflare_security.html).

# FLARE Dashboard

As mentioned in Provisioning in NVIDIA FLARE, the NVIDIA FLARE system requires a set of startup kits which include the private keys and certificates (signed by the root CA) in order to communicate to one another. The new NVFLARE Dashboard UI in NVIDIA FLARE provides a simple way to collect information of clients and users from different organizations, as well as to generate those startup kits for users to download.

Most of the details about provisioning can be found in Provisioning in NVIDIA FLARE. In this section, we focus on the user interaction with Dashboard and its backend API.


# Flower Integration

# More Examples

Federated Large Language Model (LLM)¶

        Parameter Efficient Fine Turning - Example utilizing NeMo’s PEFT methods to adapt a LLM to a downstream task.

        Prompt-Tuning Example - Example for using FLARE with NeMo for prompt learning.

        Supervised Fine Tuning (SFT) - Example to fine-tune all parameters of a LLM on supervised data.

        LLM Tuning via HuggingFace SFT Trainer - Example for using FLARE with a HuggingFace trainer for LLM tuning tasks.

Examples of NeMo-NVFlare Integration
Parameter-Efficient Fine-Tuning (PEFT) with NeMo

In this example, we utilize NeMo's PEFT using NVFlare's new Client API (minimal code changes required to run a NeMo script in FL) methods to showcase how to adapt a large language model (LLM) to a downstream task, such as financial sentiment predictions.
Supervised fine-tuning (SFT) with NeMo and NVFlare

An example of using NVIDIA FLARE with NeMo for supervised fine-tuning (SFT) to fine-tune all parameters of a large language model (LLM) on supervised data to teach the model how to follow user specified instructions.
Prompt learning with NeMo and NVFlare
An example of using NVIDIA FLARE with NeMo for prompt learning using NVFlare's Learner API to adapt a large language model (LLM) to a downstream task.


# Additional Learning Resources

Here are some additional learning resources if you will to go deeper into NVIDIA FLARE.

- NVIDIA FLARE's main webpage: [https://nvidia.github.io/NVFlare](https://nvidia.github.io/NVFlare). Here you can an introductory video to FLARE, as well as useful links to tutorials, blogs and research publication from FLARE teams.
- Sources code are hosted on GitHub: [https://github.com/NVIDIA/NVFlare](https://github.com/NVIDIA/NVFlare). Feel free to regularly watch the repository for [releases](https://github.com/NVIDIA/NVFlare/releases), [issues](https://github.com/NVIDIA/NVFlare/issues), [pull requests](https://github.com/NVIDIA/NVFlare/pulls) and [discussions](https://github.com/NVIDIA/NVFlare/discussions).

FLARE's [official documentation](https://nvflare.readthedocs.io/en/main/index.html) is also a good place to learn about further details including advanced features, APIs and best practices.

NVIDIA FLARE includes a comprehensive [catalog of tutorials and examples](https://nvidia.github.io/NVFlare/catalog/). You can pull these examples and run them yourself for further learning. Some of the examples and tutorials also include jupyter notebooks.

Over recent years, FLARE's engineering and research teams have published many [technical blogs](https://developer.nvidia.com/blog/tag/federated-learning) and [research articles](https://nvflare.readthedocs.io/en/main/publications_and_talks.html), feel free to browse them to learn more.

Lastly, as with other open-source projects, FLARE welcomes contributions from the open-source community. When you're ready, feel free to read the [contribution guidline](https://github.com/NVIDIA/NVFlare/blob/main/CONTRIBUTING.md) and start contributing!


# Recap

Let recap what we've learned in this chapter:
- We had an introductory overview of some advanced topics in FLARE including security features (privacy preserving technologies, site policy management and Confidential Computing), FLARE Dashboard, and Federated LLMs.
- Then we introduced additional links to further learning resources for you to go deeper in FLARE's design, towards becoming a FLARE developer.

# Closing Remark
That's it, we have officially finished the main content of the course "5 Minutes to Federated Learning", congratulations!

We have prepared some industry examples showing real-world federated applications using FLARE in various sectors, including [healthcare](Example_1_Medical_Imaging.ipynb), [finance](Example_2_Financial_Services_Fraud_Detection.ipynb) and [autonomous vehicles](Example_3_Autonomous_Vehicles_Cross_Country_Training.ipynb). Feel free to walk through these examples to learn more about FLARE's adoption.